In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/patient_data.csv")

In [ ]:
df.head(20)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.rename(columns={"C":"Gender"},inplace=True)

In [ ]:
# Inconsistency Corrections (clean version)

df['TakeMedication'] = df['TakeMedication'].replace({'Yes ': 'Yes'})

df['NoseBleeding'] = df['NoseBleeding'].replace({'No ': 'No'})

df['Systolic'] = df['Systolic'].replace({'121- 130': '121 - 130'})

df['Stages'] = df['Stages'].replace({'HYPERTENSIVE CRISI': 'HYPERTENSIVE CRISIS'})



In [ ]:
df.head(50)

In [ ]:
# Check how many duplicate rows exist
print(df.duplicated().sum())

# Remove duplicate rows
df.drop_duplicates(inplace=True)

In [ ]:
# Define nominal features
nominal_features = ['Gender','History','Patient','TakeMedication','BreathShortness','VisualChanges','NoseBleeding','ControlledDiet']

# Identify ordinal features
ordinal_features = [col for col in df.columns if col not in nominal_features]
ordinal_features.remove('Stages')

print(nominal_features)
print(ordinal_features)

# Convert binary Yes/No to 0/1
for col in nominal_features:
    if set(df[col].unique()) == set(['Yes','No']):
        df[col] = df[col].map({'No':0, 'Yes':1})

# Encode Gender
df['Gender'] = df['Gender'].map({'Male':0, 'Female':1})

# Encode Age groups
df['Age'] = df['Age'].map({'18-34':1, '35-50':2, '51-64':3, '65+':4})

# Encode Severity
df['Severity'] = df['Severity'].replace({'Mild':0,'Moderate':1,'Sever':2})

# Encode diagnosis duration
df['Whendiagnoused'] = df['Whendiagnoused'].map({'<1 Year':1,'1 - 5 Years':2,'>5 Years':3})

# Encode Systolic ranges
df['Systolic'] = df['Systolic'].map({'90 - 110':0,'111 - 120':1,'121 - 130':2,'130+':3})

# Encode Diastolic ranges
df['Diastolic'] = df['Diastolic'].map({'70 - 80':0,'81 - 90':1,'91 - 100':2,'100+':3})

# Encode target variable (Stages)
df['Stages'] = df['Stages'].map({
    'NORMAL':0,
    'HYPERTENSION (Stage-1)':1,
    'HYPERTENSION (Stage-2)':2,
    'HYPERTENSIVE CRISIS':3
})

In [ ]:
df.select_dtypes(include='object').columns

In [ ]:
df.dtypes

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[ordinal_features] = scaler.fit_transform(df[ordinal_features])

In [ ]:
# Gender Distribution Count Plot
plt.figure(figsize=(8,4))

sns.countplot(data=df, x="Gender", hue="Gender", palette="Set2", legend=False)

plt.title("Gender Distribution")
plt.xlabel("Gender (0 = Male, 1 = Female)")
plt.ylabel("Count")

plt.show()

In [ ]:
# Gender Distribution Pie Chart

df['Gender'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    figsize=(5,5),
    colors=['#66b3ff','#99ff99']
)

plt.title("Gender Distribution (Pie Chart)")
plt.ylabel("")

plt.show()

In [ ]:
# Hypertension Stages Distribution (show labels without changing data)

plt.figure(figsize=(8,4))

sns.countplot(data=df, x="Stages", hue="Stages", palette="coolwarm", legend=False)

plt.title("Hypertension Stages Distribution")
plt.xlabel("Hypertension Stage")
plt.ylabel("Count")

plt.xticks(
    ticks=[0,1,2,3],
    labels=[
        "NORMAL",
        "HYPERTENSION (Stage-1)",
        "HYPERTENSION (Stage-2)",
        "HYPERTENSIVE CRISIS"
    ],
    rotation=30
)

plt.show()

In [ ]:
df[['Systolic','Diastolic']].corr()

In [ ]:
# Correlation between Systolic and Diastolic Pressure

plt.figure(figsize=(6,4))

sns.heatmap(
    df[['Systolic','Diastolic']].corr(),
    annot=True,
    cmap="Blues"
)

plt.title("Correlation between Systolic & Diastolic")
plt.show()

In [ ]:
# TakeMedication vs Severity

plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x="TakeMedication",
    hue="Severity",
    palette="Set1"
)

plt.title("TakeMedication vs Severity")

plt.xlabel("TakeMedication (0 = No, 1 = Yes)")
plt.ylabel("Count")

plt.legend(title="Severity", labels=["Mild","Moderate","Severe"])

plt.show()

In [ ]:
# Age Group vs Hypertension Stages

plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x="Age",
    hue="Stages",
    palette="husl"
)

plt.title("Age Group vs Hypertension Stages")

plt.xlabel("Age Group")
plt.ylabel("Count")

plt.xticks(
    ticks=[0,1,2,3],
    labels=["18-34","35-50","51-64","65+"]
)

plt.legend(
    title="Stages",
    labels=[
        "NORMAL",
        "HYPERTENSION (Stage-1)",
        "HYPERTENSION (Stage-2)",
        "HYPERTENSIVE CRISIS"
    ]
)

plt.show()

In [ ]:
# Pairplot: Systolic vs Diastolic across Stages

sns.pairplot(
    df[['Systolic','Diastolic','Stages']],
    hue='Stages',
    diag_kind='kde',
    palette='husl'
)

plt.suptitle("Pairplot: Systolic vs Diastolic across Stages", y=1.02)

plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Stages', axis=1)
y = df['Stages']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy={}

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()

In [ ]:
X = df.drop('Stages', axis=1)
y = df['Stages']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy['Logistic Regression'] = accuracy_score(y_test, y_pred)

print("Logistic Regression Accuracy:", accuracy['Logistic Regression'])

In [ ]:
model = DecisionTreeClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy['Decision Tree'] = accuracy_score(y_test, y_pred)

print("Decision Tree Accuracy:", accuracy['Decision Tree'])

In [ ]:
model = RandomForestClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy['Random Forest'] = accuracy_score(y_test, y_pred)

print("Random Forest Accuracy:", accuracy['Random Forest'])

In [ ]:
model = SVC()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy['SVM'] = accuracy_score(y_test, y_pred)

print("SVM Accuracy:", accuracy['SVM'])

In [ ]:
model = KNeighborsClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy['KNN'] = accuracy_score(y_test, y_pred)

print("KNN Accuracy:", accuracy['KNN'])

In [ ]:
model = GaussianNB()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy['Naive Bayes'] = accuracy_score(y_test, y_pred)

print("Naive Bayes Accuracy:", accuracy['Naive Bayes'])

In [ ]:
model = RidgeClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy['Ridge Classifier'] = accuracy_score(y_test, y_pred)

print("Ridge Classifier Accuracy:", accuracy['Ridge Classifier'])

In [ ]:
for model, acc in accuracy.items():
    print(model, ":", acc)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Final model
final_model = LogisticRegression(max_iter=1000)

final_model.fit(X_train, y_train)

In [ ]:
y_pred = final_model.predict(X_test)

from sklearn.metrics import accuracy_score

print("Final Model Accuracy:", accuracy_score(y_test, y_pred))
model = joblib.load("model/hypertension_model.pkl")

In [ ]:
import joblib

joblib.dump(final_model, "hypertension_model.pkl")